In [ ]:
#imports for the project
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns  
from PIL import Image

Configures the data set
Creates a dictionary with the data set, number of classes, class names, image size, learning rate, batch size, and number of epochs.

In [ ]:
CONFIG = {
    "data_dir": "/kaggle/input/datasets/masonrbrito/pimples-2/data/Pimple Dataset",
    "num_classes": 6,
    "class_names": ["blackheads", "cysts","nodules", "papules", "pustules", "whiteheads"],
    "img_size": 224,
    "batch_size": 32,     
    "num_epochs": 30,      
    "learning_rate": .004,   
    "save_path": "/kaggle/working/best_model.pth",
    "device": "cuda" if torch.cuda.is_available() else "cpu",
}

Creates the method in order to take in data.

In [ ]:
#Data collection
def get_transforms(config):
    """
    Define transforms for training and validation sets.
    """
    #For normalize we used mean = [.485, .456, .406] and std = [.229, .224, .225]
    #because those values are common for image processing

    #Use .Compose() to pack all image transformations we want to train the data
    #Give each image a random flip and rotation
    train_transform = transforms.Compose([
        transforms.Resize((config["img_size"], config["img_size"])),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        transforms.RandomRotation(30),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ])
    val_transform = transforms.Compose([
        transforms.Resize((config["img_size"], config["img_size"])),  
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ])

    return train_transform, val_transform


def get_dataloaders(config):
    """
    Create and return DataLoader objects for train, val, and test sets.
    """
    train_transform, val_transform = get_transforms(config)

    #Get the datasets, with the normalized size and randomized rotations for train data
    train_dataset = datasets.ImageFolder(os.path.join(config['data_dir'], 'train'), transform=train_transform)
    val_dataset = datasets.ImageFolder(os.path.join(config['data_dir'],'val'),   transform=val_transform)
    test_dataset = datasets.ImageFolder(os.path.join(config['data_dir'], 'test'),  transform=val_transform)

    #Shuffle and load the datasets, and get 224 of each 
    train_loader = DataLoader(train_dataset, batch_size=config["batch_size"], shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=config["batch_size"], shuffle=False)  
    test_loader = DataLoader(test_dataset, batch_size=config["batch_size"], shuffle=False)
    return train_loader, val_loader, test_loader

Creates model based on the configuration given above.

In [ ]:
def build_model(config, freeze_backbone=True):
    """
    TODO: Load a pretrained ResNet-50 and modify it for our task.
    - Load ResNet-50 with pretrained ImageNet weights
    - If freeze_backbone is True, freeze all layers (for baseline run)
    - Replace the final fully connected layer to output config["num_classes"] classes
    - Move model to config["device"]
    Hint: model.fc is the final layer, model.fc.in_features gives you the input size
    """
    #Intialize the resnet50 model
    model = models.resnet50(weights= models.ResNet50_Weights.IMAGENET1K_V1)

    if freeze_backbone:
        for param in model.parameters():
            #Turn off the gradients for each layer
            param.requires_grad = False

    model.fc = nn.Linear(model.fc.in_features, config['num_classes'])

    model = model.to(config['device'])
    return model

In [ ]:
def build_baseline(config):

    #intialize the baseline model (resnet 18)
    model = models.resnet18(weights= models.ResNet18_Weights.IMAGENET1K_V1)

    model.fc = nn.Linear(model.fc.in_features, config["num_classes"])

    model = model.to(config['device'])
    return model

methods in order to train the model.

In [ ]:
def train_one_epoch(model, loader, loss_function, optimizer, device):
    """
    Run one full pass over the training data.
    """
    #Set the model to train mode
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = loss_function(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    avg_loss = running_loss / total
    accuracy = correct / total
    return avg_loss, accuracy


def evaluate(model, loader, loss_function, device):
    """
    Evaluate the model on a given dataloader (val or test).
    """
    #Put the model in evaluation mode
    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = loss_function(outputs, labels)

            running_loss += loss.item() * images.size(0)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_loss = running_loss / total
    accuracy = correct / total
    return avg_loss, accuracy, all_preds, all_labels

Methods to plot the data in order to see how our training did compared to the actual pictures

In [ ]:
                       
def plot_loss_curves(history):
    """
    Plot training and validation loss curves on the same graph.
    """
    epochs = range(1, len(history["train_loss"]) + 1)

    plt.figure(figsize=(8, 5))
    plt.plot(epochs, history["train_loss"], label="Train Loss", marker="o")
    plt.plot(epochs, history["val_loss"], label="Val Loss", marker="o")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Training vs Validation Loss")
    plt.legend()
    plt.tight_layout()
    plt.show()

def plot_confusion_matrix(preds, labels, class_names):
    """
    Plot a confusion matrix from predictions and true labels.
    """
    cm = confusion_matrix(labels, preds)

    plt.figure(figsize=(8, 6))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=class_names,
        yticklabels=class_names,
    )
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title("Confusion Matrix")
    plt.tight_layout()
    plt.show()


def plot_accuracy_curves(history):
    epochs = range(1, len(history["train_acc"]) + 1)
    plt.figure(figsize=(8, 5))
    plt.plot(epochs, history["train_acc"], label="Train Acc", marker="o")
    plt.plot(epochs, history["val_acc"], label="Val Acc",   marker="o")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title("Training vs Validation Accuracy")
    plt.legend(); plt.tight_layout(); plt.show()


def plot_per_class_f1(preds, labels, class_names):
    report = classification_report(labels, preds, target_names=class_names, output_dict=True)
    f1_scores = [report[cls]["f1-score"] for cls in class_names]
   
    plt.figure(figsize=(8, 5))
    bars = plt.bar(class_names, f1_scores, color="steelblue")
    plt.ylim(0, 1)
    plt.xlabel("Class")
    plt.ylabel("F1 Score")
    plt.title("Per-Class F1 Score")
    for bar, score in zip(bars, f1_scores):
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f"{score:.2f}", ha="center", fontsize=10)
    plt.tight_layout(); plt.show()


Main method in order to run the methods

In [ ]:
def predict_image(model,image_path, config):
    #Get default transformation
    _, val_transformation = get_transforms(config)
    #Open the image and convert it to RGB
    image = Image.open(image_path).convert("RGB")
    image = val_transformation(image).unsqueeze(0).to(config['device'])

    model.eval()
    with torch.no_grad():
        output = model(image)
        probs = torch.softmax(output, dim=1)
        pred_idx = torch.argmax(probs, dim=1)
        confidence = probs[0][pred_idx].item()
    pred_class = config["class_names"][pred_idx.item()]
    print(f"Prediction: {pred_class}({confidence:.2%} confidence)")
    return pred_class, confidence
    
    

In [ ]:
def advice(pred_class):
    if(pred_class == 'blackheads'or pred_class == 'papules' or pred_class == 'pustules'):
        print("Use Salicylic acid and retinoid cream to treat it")
    if(pred_class == 'cysts'):
        print("You can pop, but better option is to continue washing face and let it dry out naturally")
    if(pred_class == 'nodules'):
        print("You can use Benzoyl Peroxide to treat the it")
    if(pred_class == 'whiteheads'):
        print('If whitehead is popped squeeze and keep it clean. If it is not let it dry out naturally')
        
    
    

In [ ]:
##Run the basline model
train_loader, val_loader, test_loader = get_dataloaders(CONFIG)

model = build_baseline(CONFIG)
loss_function = nn.CrossEntropyLoss()
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()),
                       lr=CONFIG["learning_rate"])

history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

for epoch in range(CONFIG['num_epochs']):
        train_loss, train_acc = train_one_epoch(model, train_loader, loss_function, optimizer, CONFIG["device"])
        val_loss, val_acc, _, _ = evaluate(model, val_loader, loss_function, CONFIG["device"])

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        print(f"Epoch {epoch+1}/{CONFIG['num_epochs']} | "
              f"Train Loss: {train_loss:.4f}  Acc: {train_acc:.4f} | "
              f"Val Loss: {val_loss:.4f}  Acc: {val_acc:.4f}")



In [ ]:
if __name__ == "__main__":
    train_loader, val_loader, test_loader = get_dataloaders(CONFIG)

    # Step 1: Baseline — frozen backbone
    model = build_model(CONFIG, freeze_backbone=True)
    loss_function = nn.CrossEntropyLoss()
    optimizer = optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=CONFIG["learning_rate"]
    )

    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

    for epoch in range(CONFIG["num_epochs"]):
        train_loss, train_acc = train_one_epoch(model, train_loader, loss_function, optimizer, CONFIG["device"])
        val_loss, val_acc, _, _ = evaluate(model, val_loader, loss_function, CONFIG["device"])

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        print(f"Epoch {epoch+1}/{CONFIG['num_epochs']} | "
              f"Train Loss: {train_loss:.4f}  Acc: {train_acc:.4f} | "
              f"Val Loss: {val_loss:.4f}  Acc: {val_acc:.4f}")

    # Step 2: Fine-tuning — unfreeze backbone, lower learning rate
    model = build_model(CONFIG, freeze_backbone=False)
    optimizer = optim.Adam(model.parameters(), lr=CONFIG["learning_rate"] / 10)
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

    for epoch in range(CONFIG["num_epochs"]):
        train_loss, train_acc = train_one_epoch(model, train_loader, loss_function, optimizer, CONFIG["device"])
        val_loss, val_acc, _, _ = evaluate(model, val_loader, loss_function, CONFIG["device"])

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        torch.save(model.state_dict(), CONFIG['save_path'])

        print(f"[Fine-tune] Epoch {epoch+1}/{CONFIG['num_epochs']} | "
              f"Train Loss: {train_loss:.4f}  Acc: {train_acc:.4f} | "
              f"Val Loss: {val_loss:.4f}  Acc: {val_acc:.4f}")

    # Step 3: Evaluate on test set and plot results
    test_loss, test_acc, test_preds, test_labels = evaluate(model, test_loader, loss_function, CONFIG["device"])
    print(f"\nTest Accuracy: {test_acc:.4f}")
    

In [ ]:
model.load_state_dict(torch.load(CONFIG["save_path"]))
image_path = r"/kaggle/input/datasets/masonrbrito/pimple-dataset/valid/acne_228_jpg.rf.oMOxgS1lBNh5vJuHJmYM.jpg"
predict_image(model, image_path, CONFIG)

In [ ]:
plot_loss_curves(history)
plot_confusion_matrix(test_preds, test_labels, CONFIG["class_names"])
plot_accuracy_curves(history)
plot_per_class_f1(test_preds, test_labels, CONFIG["class_names"])

In [ ]:
model.load_state_dict(torch.load(CONFIG["save_path"]))
image_path = r"/kaggle/input/datasets/masonrbrito/pimple-dataset/valid/acne_415_jpg.rf.SCuyXPgqGIRhyCu6kI4N.jpg"
pred_class, _ = predict_image(model, image_path, CONFIG)

advice(pred_class)

In [ ]:
model.load_state_dict(torch.load(CONFIG["save_path"]))
image_path = r"/kaggle/input/datasets/masonrbrito/pimple-dataset/valid/acne_228_jpg.rf.oMOxgS1lBNh5vJuHJmYM.jpg"
pred_class, _ = predict_image(model, image_path, CONFIG)

advice(pred_class)

